# Day 50: Project 07 Kick-off: an ML Predictor in Miniature

**Today's question:** what does a *complete*, *honest* ML project look like from start to finish?

This notebook walks through every section the Project 07 brief asks for, on a dataset **nobody** will use for the
project: the iris flowers. Your project follows the same sections with your own data and your own reasoning.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

## 1. Problem statement

> A community garden labels its iris plants by hand, and volunteers often mix up the species. I predict the
> **species** (setosa, versicolor, virginica) from four measurements a volunteer can take with a ruler.
> **Classification**, 3 classes, target `target`. Useful result: clearly better than the baseline, with very few
> mix-ups, because a wrong label only means a wrong sign in the garden.

Notice what it contains: *what*, *for whom*, *how it is used*, *task type*, *what "good" means*.

## 2. Load and look at the data

In [ ]:
iris = load_iris(as_frame=True)
flowers = iris.frame
print(flowers.shape)
print(iris.target_names)
flowers.head()

In [ ]:
flowers["target"].value_counts()

150 flowers, exactly 50 of each species (balanced classes), no missing values.

## 3. Features, target and leakage check

| Feature | Why it might help | Known at prediction time? |
|---------|-------------------|---------------------------|
| petal length | species have visibly different petal sizes | yes, measured with a ruler |
| petal width | same | yes |
| sepal length | overall flower size | yes |
| sepal width | shape of the flower | yes |

Left out: nothing else exists. Leakage check: every feature is measured **before** anyone knows the species. If the
file had a column like "label written on the sign", it would leak the answer.

In [ ]:
FEATURES = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
TARGET = "target"

X = flowers[FEATURES]
y = flowers[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("training rows:", len(X_train), " test rows:", len(X_test))

## 4. Baseline, two models, one results table

The pattern you will reuse: a small function that fits one model and returns a dict of its results, then a list of
dicts becomes a table.

In [ ]:
def evaluate(name, model):
    """Fit the model on the training data and return its training and test accuracy."""
    model.fit(X_train, y_train)
    return {
        "model": name,
        "train_accuracy": round(model.score(X_train, y_train), 3),
        "test_accuracy": round(accuracy_score(y_test, model.predict(X_test)), 3),
    }


baseline = DummyClassifier(strategy="most_frequent")
log_model = LogisticRegression(max_iter=1000)
tree_model = DecisionTreeClassifier(max_depth=2, random_state=42)

results = [
    evaluate("baseline (most common species)", baseline),
    evaluate("logistic regression", log_model),
    evaluate("decision tree, depth 2", tree_model),
]
pd.DataFrame(results)

## 5. The metric in plain words

**Accuracy** fits here: three balanced classes and no mistake is much worse than another.

> Guessing the most common species is right for only 1 flower in 3 (0.29 on this test set). Logistic regression
> labels every one of the 38 test flowers correctly; the depth-2 tree gets 37 of 38.

With only 38 test flowers, "100%" means "no mistakes on 38 flowers", not "never wrong".

## 6. A plot that shows the mistakes

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, tree_model.predict(X_test), display_labels=iris.target_names, cmap="Blues"
)
plt.title("Depth-2 tree: one versicolor labelled as virginica")
plt.show()

## 7. One improvement, reported honestly

Plan made in advance: the tree's depth (2) was a guess, so I choose it with cross-validation on the **training**
data (not by peeking at the test set), then measure the chosen tree on the test set once.

In [ ]:
best_depth = 0
best_cv = 0
for depth in [1, 2, 3, 4, 5]:
    scores = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=42), X_train, y_train, cv=5)
    print("depth", depth, "average CV accuracy:", round(scores.mean(), 3))
    if scores.mean() > best_cv:
        best_depth = depth
        best_cv = scores.mean()
print("chosen depth:", best_depth)

In [ ]:
results.append(evaluate(f"decision tree, depth {best_depth} (tuned)",
                        DecisionTreeClassifier(max_depth=best_depth, random_state=42)))
pd.DataFrame(results)

Write the result in one honest sentence, whatever it is: *"Cross-validation chose depth 4, which fixed the tree's
one mistake and now ties with logistic regression (38 of 38). I keep logistic regression: it was already perfect on
the test set and needed no tuning."* If the improvement had made things worse, I would say so just as plainly and
keep the original.

## 8. Using the model: `predict_one`

In [ ]:
final_model = log_model


def predict_one(example):
    """Return the predicted species name for one flower given as a dict of measurements."""
    missing = [name for name in FEATURES if name not in example]
    if missing:
        raise ValueError(f"missing measurements: {missing}")
    table = pd.DataFrame([example])[FEATURES]      # same columns, same order as in training
    species_number = final_model.predict(table)[0]
    return iris.target_names[species_number]

In [ ]:
mystery_flower = {
    "sepal length (cm)": 6.1, "sepal width (cm)": 2.8,
    "petal length (cm)": 4.7, "petal width (cm)": 1.2,
}
print("Prediction:", predict_one(mystery_flower))

In [ ]:
# A missing measurement must give a clear error, not a confusing one from deep inside scikit-learn
predict_one({"petal length (cm)": 4.7})

## 9. Limitations & responsible use

- 150 flowers from one historical collection; the test set has 38. Other gardens, varieties or hybrids may differ.
- The model always answers one of three species, even for a rose. It cannot say "I don't know this flower".
- Measurements by different volunteers vary; a careless measurement can flip a borderline flower.
- Low stakes here: a wrong sign. In your project, think about **who** is affected by each kind of mistake.

## 10. Your turn

Open `projects/07-ml-predictor/starter/ml_predictor.ipynb`. Same sections, your data:

| Option | Data | Task | Main metric |
|--------|------|------|-------------|
| a | `students.csv` → `passed` | classification | accuracy + recall/precision for "fail" |
| b | `students.csv` → `final_score` | regression | MAE (points) |
| c | `apartments.csv` → `price_k` | regression | MAE (thousands) |
| d | `load_wine` / `load_breast_cancer` (ask first) | classification | accuracy (+ recall for the class that matters) |